# Human-in-the-loop evaluation solution

**Executed synthetic demo.** This notebook illustrates the complete code/configuration/analysis workflow. No real Space, provider outputs, human annotations or colleague review were available. The20 reference-labelled energy responses are explicitly synthetic. Genuine completion requires the live steps below.

**Execution method:** Code cells were run sequentially in a shared Python process with captured text/table outputs. Native Jupyter kernel startup is blocked by this environment's socket restrictions. This verifies the cell code, not Jupyter startup. README is the run/submission guide.

In [ ]:
%pip install pandas

In [ ]:
from pathlib import Path
import sys,json,inspect
import pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'src').exists() and (ROOT.parent/'src').exists():ROOT=ROOT.parent
assert (ROOT/'src/workflow.py').exists(), 'Open from this lab root or its notebooks folder.'
sys.path.insert(0,str(ROOT))
from src.workflow import (load_json,validate_records,config_description,make_settings,
    flatten_annotations,analyze,run_analysis,connect,create_dataset,upload_records,
    get_dataset,snapshot_records)
from src.make_demo import generate_demo
RUN_LIVE=False  # Requires actual Space/credentials and real user annotations.
print('Synthetic offline demonstration; RUN_LIVE =',RUN_LIVE)

ModuleNotFoundError: No module named 'pandas'

## Steps1-2: deploy and connect
Follow `docs/space_setup.md`: use the official Argilla template, record the real direct Space URL, configure visibility/persistence, and retrieve your key. Store credentials locally in `.env`. Private Spaces also require HF authorization. No deployment URL or login is invented here. The live cell is opt-in.

In [ ]:
if RUN_LIVE:
    client=connect()
else:
    print('Live connection skipped. Real Space, credentials and UI verification remain pending.')

Live connection skipped. Real Space, credentials and UI verification remain pending.


## Step3: prepare diverse responses
The demo has10 energy coding instructions with2 authored candidates each. This exceeds the5-10-example minimum. Each response remains separate; prompt_id joins A/B. Candidate aliases reduce model-reputation cues. Use genuine LLM outputs for the real lab, optionally via `python -m src.generate_responses`. That command is separate, requires OpenAI credentials and incurs cost.

In [ ]:
records,payload=generate_demo()
validate_records(records)
print('Response records:',len(records),'| paired prompts:',len(set(x['prompt_id'] for x in records)))
display(pd.DataFrame(records)[['record_id','prompt_id','candidate_alias','topic','data_origin']].head(8))
print(records[0]['instruction'])
print(records[0]['response'])

Response records: 20 | paired prompts: 10


,record_id,prompt_id,candidate_alias,topic,data_origin
0,E02-B,E02,candidate_B,mean_load,synthetic_authored_demo
1,E01-A,E01,candidate_A,energy_units,synthetic_authored_demo
2,E09-A,E09,candidate_A,percentage_change,synthetic_authored_demo
3,E10-B,E10,candidate_B,load_factor,synthetic_authored_demo
4,E05-A,E05,candidate_A,battery_runtime,synthetic_authored_demo
5,E04-A,E04,candidate_A,capacity_factor,synthetic_authored_demo
6,E01-B,E01,candidate_B,energy_units,synthetic_authored_demo
7,E03-A,E03,candidate_A,peak_load,synthetic_authored_demo


Write mean_load(values) for a nonempty list of nonnegative numeric kW samples. Reject an empty list or negative sample with ValueError. Include a docstring and two examples.
```python
def mean_load(values):
    """Return mean kW load; empty or negative samples raise ValueError."""
    if not values or any(v < 0 for v in values):
        raise ValueError("nonempty nonnegative samples required")
    return sum(values) / len(values)

# mean_load([2, 4]) -> 3.0
# mean_load([]) raises ValueError
```


## Step4: annotator guidelines
Read the full root guideline document. Code0-3 + documentation0-2 + examples0-2 forms an additive0-7 score. The three scores stay separate. A colleague must review clarity and an independent3-record pilot should resolve ambiguous anchors before the final human annotation run. The included review record remains pending.

In [ ]:
print((ROOT/'annotator_guidelines.md').read_text())

# Annotator guidelines: energy Python response evaluation

Version1.0, 10 October2026. Scope: small Python arithmetic/data helpers for an energy consulting workflow. **Guideline clarity review by a colleague is pending.** Record the review and any revision in `docs/guideline_review.md` before collecting final labels.

## Task

Read the instruction, candidate response and validation context. Judge whether the response meets that specific programming contract, and give separate scores for code, documentation and examples. These helpers are teaching examples, not authorization for engineering or operational decisions.

## Annotation sequence

1. Read the instruction first. Note function name, units, boundaries and required exception behavior.
2. Read the response without consulting other annotators, synthetic reference scores or automated suggestions. Candidate aliases are anonymous; do not infer which model is better.
3. Inspect the code and check at least one ordinary and one edge input

## Steps5-6: configure and upload
Three required Markdown fields, six required questions, four metadata filters, one submitted annotation per record by default. Real settings use the Argilla2.8 SDK. No synthetic reference annotation is uploaded by the live upload function. Existing unchanged IDs are reused; changed content is rejected to protect existing labels. Test2 records in the UI before uploading all.

In [ ]:
config=config_description()
display(pd.DataFrame(config['questions']))
print('Fields:',[x['name'] for x in config['fields']])
print(inspect.getsource(make_settings))
if RUN_LIVE:
    live_records=validate_records(load_json(ROOT/'data/live/evaluation_records.json'))
    # Dataset creation is explicit; fails safely if the name already exists.
    dataset=create_dataset(client,min_submitted=1)
    upload_records(dataset,live_records,limit=2)
    print('Inspect pilot fields/questions and submit genuine annotations in the UI. Full upload is a separate CLI step.')

,name,type,values,required,labels
0,code_quality,RatingQuestion,"[0, 1, 2, 3]",True,NaN
1,documentation,RatingQuestion,"[0, 1, 2]",True,NaN
2,examples,RatingQuestion,"[0, 1, 2]",True,NaN
3,primary_issue,LabelQuestion,NaN,True,"[none, correctness, missing_validation, missing_documentation, missing_examples, mixed, uncertain]"
4,evaluation_confidence,LabelQuestion,NaN,True,"[high, low]"
5,comment,TextQuestion,NaN,True,NaN


Fields: ['instruction', 'response', 'validation_context']
def make_settings(min_submitted=1):
    """Call after establishing a real Argilla client; SDK resources use its default."""
    import argilla as rg
    config_description(min_submitted)
    return rg.Settings(
        guidelines=(ROOT/'annotator_guidelines.md').read_text(encoding='utf-8'),
        fields=[rg.TextField(name=f, required=True, use_markdown=True) for f in FIELDS],
        questions=[
            rg.RatingQuestion(name='code_quality', values=[0,1,2,3], title='Code correctness and robustness (0-3)',
                description='0 unusable; 1 major functional errors; 2 minor limitations; 3 correct and robust on specified checks.'),
            rg.RatingQuestion(name='documentation', values=[0,1,2], title='Documentation (0-2)',
                description='0 none; 1 partial; 2 clear docstring and useful explanation of inputs, units and invalid cases.'),
            rg.RatingQuestion(name='examples', values=[0,1,2], tit

## Step7: human annotation
For the real run, rate each answer independently in the Argilla UI. Inspect ordinary/edge behavior, score all components, give a concrete comment and **Submit**. Drafts do not count. No Python cell can substitute for actual human judgment. Do not consult reference fixtures or model suggestions while forming independent labels. The demo source below is not human data.

In [ ]:
print('Annotation source:',payload[0]['annotation_source'])
print('Demo fixture identity:',payload[0]['responses'][0]['user_id'])
print('Real colleague review and actual human annotations: pending.')

Annotation source: synthetic_reference_not_human
Demo fixture identity: synthetic_reference_fixture
Real colleague review and actual human annotations: pending.


## Step8: export and validate
Preserve each question value, user ID and submission status. Join by record+user, not row order. Complete submitted responses enter the normalized CSV; drafts/incomplete/invalid scores are excluded and audited. Suggestions never become human responses just because they appear in an export.

In [ ]:
annotations,rejected=flatten_annotations(payload)
assert len(annotations)==20 and rejected.empty
assert annotations.total_score.eq(annotations[['code_quality','documentation','examples']].sum(axis=1)).all()
display(annotations[['record_id','annotator_id','code_quality','documentation','examples','total_score','annotation_source']].head())
print(inspect.getsource(snapshot_records))
# After genuine human annotation, use: python -m src.workflow export
# It replaces root CSV/JSON/summary with real user-attributed submitted results.
# Do not rerun demo afterward unless you want root exports changed back to demo.

,record_id,annotator_id,code_quality,documentation,examples,total_score,annotation_source
0,E01-A,synthetic_reference_fixture,0,0,0,0,synthetic_reference_not_human
1,E01-B,synthetic_reference_fixture,3,2,2,7,synthetic_reference_not_human
2,E02-A,synthetic_reference_fixture,1,0,1,2,synthetic_reference_not_human
3,E02-B,synthetic_reference_fixture,3,2,2,7,synthetic_reference_not_human
4,E03-A,synthetic_reference_fixture,3,2,2,7,synthetic_reference_not_human


def snapshot_records(dataset):
    """Preserve per-user response status; to_list() alone omits it in SDK2.8."""
    result=[]
    for record in dataset.records(with_responses=True, with_suggestions=True):
        responses=[]
        for response in record.responses:
            responses.append({'question_name':response.question_name, 'value':response.value,
                              'user_id':str(response.user_id),
                              'status':getattr(response.status,'value',response.status)})
        result.append({'record_id':str(record.id),'server_record_id':str(record._server_id or ''),
            'fields':dict(record.fields),'metadata':dict(record.metadata),
            'responses':responses, 'suggestions':[s.serialize() for s in record.suggestions],
            'annotation_source':'argilla_user_response', 'exported_at':datetime.now(timezone.utc).isoformat()})
    return result



## Step9: analyze and produce the required paragraph
Record means prevent extra raters from giving a response extra weight. Candidate comparison uses matched prompts only. Histograms count annotation rows and identify that denominator. Agreement is null when no shared multi-rater records exist. Demo fixture statistics cannot establish a model ranking or human reliability.

In [ ]:
metrics=run_analysis(payload,ROOT/'outputs/demo','demo',write_root=True)
display(pd.read_csv(ROOT/'outputs/demo/candidate_statistics.csv'))
display(pd.read_csv(ROOT/'outputs/demo/paired_comparison.csv'))
print((ROOT/'lab_summary.md').read_text())
assert metrics['annotated_records']==20
assert metrics['paired_comparison']['matched_prompt_count']==10
assert metrics['exact_total_agreement'] is None
assert not metrics['human_annotations_independently_verified']

{
  "annotation_rows": 20,
  "annotated_records": 20,
  "source_records": 20,
  "coverage": 1.0,
  "annotators": 1,
  "overall_record_weighted": {
    "mean": 4.25,
    "median": 5.0,
    "min": 0.0,
    "max": 7.0
  },
  "mean_components_record_weighted": {
    "code_quality": 1.9,
    "documentation": 1.15,
    "examples": 1.2
  },
  "paired_comparison": {
    "matched_prompt_count": 10,
    "mean_A_minus_B": -0.3,
    "A_wins": 5,
    "B_wins": 5,
    "ties": 0
  },
  "annotation_score_distribution": {
    "0": 4,
    "1": 1,
    "2": 1,
    "3": 4,
    "7": 10
  },
  "primary_issue_counts_annotation_weighted": {
    "none": 10,
    "correctness": 5,
    "missing_validation": 4,
    "mixed": 1
  },
  "low_confidence_annotation_rows": 0,
  "multi_annotated_records": 0,
  "exact_total_agreement": null,
  "data_origins": [
    "synthetic_authored_demo"
  ],
  "annotation_sources": [
    "synthetic_reference_not_human"
  ],
  "mode": "demo",
  "excluded_response_groups": 0,
  "deploymen

,candidate_alias,count,mean,median,min,max
0,candidate_A,10,4.1,5.0,0.0,7.0
1,candidate_B,10,4.4,5.0,0.0,7.0


,prompt_id,candidate_A,candidate_B,difference_A_minus_B
0,E01,0.0,7.0,-7.0
1,E02,2.0,7.0,-5.0
2,E03,7.0,0.0,7.0
3,E04,0.0,7.0,-7.0
4,E05,3.0,7.0,-4.0
5,E06,7.0,3.0,4.0
6,E07,1.0,7.0,-6.0
7,E08,7.0,0.0,7.0
8,E09,7.0,3.0,4.0
9,E10,7.0,3.0,4.0


The synthetic demonstration contains 20 complete annotation rows across 20 of 20 response records (100% coverage); the record-weighted mean is 4.25/7 (median 5.00, range 0-7). Across 10 matched prompts, candidate A wins 5, candidate B wins 5, and 0 tie; the demo fixtures illustrate how correct edge-case handling, documentation and examples affect the rubric; these are synthetic reference labels, so no human/model performance or inter-annotator reliability is established. Next, obtain real model outputs, have a colleague review the guidelines, collect blind human ratings in Argilla and replace this demonstration with a live export.



## Reflection and actual submission
Read `reflection.md`, `docs/requirements_checklist.md` and `docs/live_evidence.md`. Genuine completion still needs an Argilla Space, real LLM outputs, colleague-reviewed guidelines, human annotation, live export and a dedicated GitHub repository URL. Required root files are annotator_guidelines.md, dataset_config.md, annotated_dataset.csv/JSON, lab_summary.md and README. The PDF/HTML report remains demo evidence after a live CLI run.

Sources: [Argilla quickstart](https://docs.argilla.io/latest/getting_started/quickstart/), [human annotation UI](https://docs.argilla.io/latest/how_to_guides/annotate/), [record export](https://docs.argilla.io/latest/how_to_guides/import_export/), and the supplied lab brief. Full bibliography: `docs/sources.md`.